# ResNet-18 via DaisyTuner on Q.ANT Gen2 NPU

In [ ]:
import pytest
import os
import torch
import torchvision.models as models
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image as Image
import requests
import torchvision.transforms as transforms

import docc.torch
from docc.qant import register_docc_plugin

register_docc_plugin()

docc.torch.set_backend_options(target="qant", category="server")

In [ ]:
# Load a pre-trained ResNet-18 model
model = models.resnet50(weights="IMAGENET1K_V1")
model.eval()

# Get Imagenet class labels
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(labels_url).text.splitlines()

In [ ]:
# Compile the model
print("\n--- Compile Times ---")
start = time.perf_counter()
program_ref = torch.compile(model)
torch_compile_time = time.perf_counter() - start
print(f"Torch compile: {torch_compile_time:.4f} s")


start = time.perf_counter()
program = torch.compile(model, backend="docc")
docc_compile_time = time.perf_counter() - start
print(f"Docc compile: {docc_compile_time:.4f} s")

In [ ]:
# select a image
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"  # dog (Samoyed)

image = Image.open(requests.get(url, stream=True).raw)

preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
input_tensor = preprocess(image)
input_batch = input_tensor.unsqueeze(0)  # create a mini-batch as expected by the model

_ = plt.imshow(image)
plt.title("Input Image")
plt.axis("off")

In [ ]:
# Force dynamo (inference) backend
with torch.no_grad():
    print("--- Execution Times ---")
    start = time.perf_counter()
    res = program(input_batch)
    docc_exec_time = time.perf_counter() - start
    print(f"Docc first execution: {docc_exec_time:.4f} s")

    start = time.perf_counter()
    ref = program_ref(input_batch)
    torch_exec_time = time.perf_counter() - start
    print(f"Torch first execution: {torch_exec_time:.4f} s")

    start = time.perf_counter()
    res = program(input_batch)
    docc_exec_time = time.perf_counter() - start
    print(f"Docc 2nd execution: {docc_exec_time:.4f} s")

    start = time.perf_counter()
    ref = program_ref(input_batch)
    torch_exec_time = time.perf_counter() - start
    print(f"Torch 2nd execution: {torch_exec_time:.4f} s")


_, predicted_idx = torch.max(res, 1)
print(
    f"Predicted class: {labels[predicted_idx]} with certainty {torch.nn.functional.softmax(res, dim=1)[0][predicted_idx].item():.4f}"
)
print(
    f"Reference class: {labels[torch.argmax(ref)]} with certainty {torch.nn.functional.softmax(ref, dim=1)[0][torch.argmax(ref)].item():.4f}"
)